In [1]:
import pandas as pd

df = pd.read_csv("/Users/admin/Desktop/skincare_dupe_project/data/skincare_products_raw.csv")

df.head()

,product_name,ingredients,url
0,COSRX Advanced Snail Radiance Dual Essence,"snail secretion filtrate, butylene glycol, nia...",https://incidecoder.com/products/cosrx-advance...
1,CeraVe Baby Moisturizing Lotion,"water, cetearyl alcohol, caprylic/​capric trig...",https://incidecoder.com/products/cerave-baby-m...
2,COSRX Ac Collection Calming Foam Cleanser,"water, glycerin, stearic acid, myristic acid, ...",https://incidecoder.com/products/cosrx-ac-coll...
3,The Ordinary Aha 30% + Bha 2% Peeling Solution...,"glycolic acid, aqua (water), aloe barbadensis ...",https://incidecoder.com/products/the-ordinary-...
4,The Ordinary Buffet + Copper Peptides 1%,"aqua (water), glycerin, lactococcus ferment ly...",https://incidecoder.com/products/the-ordinary-...


In [2]:
df.columns

Index(['product_name', 'ingredients', 'url'], dtype='str')

In [3]:
import re

#### Ingredient cleaning function

In [69]:
import re

def clean_ingredients(text):

    text = text.lower()

    # Fix known ingredient formatting issues
    text = text.replace("1,2-hexanediol", "1-2-hexanediol")

    # Remove text inside parentheses
    text = re.sub(r"\(.*?\)", "", text)

    # Split into ingredients
    ingredients = text.split(",")

    cleaned = []

    for ingredient in ingredients:

        ingredient = ingredient.strip()

        # Remove trailing periods
        ingredient = ingredient.rstrip(".")

        # Remove extra whitespace
        ingredient = re.sub(r"\s+", " ", ingredient)

        # Skip empty strings
        if not ingredient:
            continue

        # Remove obvious website noise
        if "read here" in ingredient:
            continue

        if "ingredient list" in ingredient:
            continue

        if "more detail" in ingredient:
            continue

        if ">>" in ingredient:
            continue

        # Remove extremely short garbage tokens
        if len(ingredient) < 2:
            continue

        cleaned.append(ingredient)

    return cleaned

#### Apply cleaning

In [70]:
df["cleaned_ingredients"] = df["ingredients"].apply(clean_ingredients)

In [71]:
df[["ingredients", "cleaned_ingredients"]].head()

,ingredients,cleaned_ingredients
0,"snail secretion filtrate, butylene glycol, nia...","[snail secretion filtrate, butylene glycol, ni..."
1,"water, cetearyl alcohol, caprylic/​capric trig...","[water, cetearyl alcohol, caprylic/​capric tri..."
2,"water, glycerin, stearic acid, myristic acid, ...","[water, glycerin, stearic acid, myristic acid,..."
3,"glycolic acid, aqua (water), aloe barbadensis ...","[glycolic acid, aqua, aloe barbadensis leaf wa..."
4,"aqua (water), glycerin, lactococcus ferment ly...","[aqua, glycerin, lactococcus ferment lysate, c..."


In [72]:
vocab = set()

for ingredients in df["cleaned_ingredients"]:
    
    for ingredient in ingredients:
        vocab.add(ingredient)

In [73]:
len(vocab)

566

In [74]:
list(vocab)[:50]

['polypodium vulgare rhizome extract',
 'ethylhexyl glycerin',
 'butyloctyl salicylate',
 'candelilla wax esters',
 'aloe barbadensis leaf juice',
 'glucose',
 'sodium lactate',
 'xanthan gum',
 'sodium polynaphthalenesulfonate',
 'stearyl alcohol',
 'melanin',
 'coco glucoside',
 'chlorhexidine dihydrochloride',
 'trisodium ethylenediamine disuccinate',
 'diethylhexyl sodium sulfosuccinate',
 'aqua',
 'camellia japonica seed oil',
 'arginine pca',
 'methylpropanediol',
 'capryloyl glycine',
 'peg-60 hydrogenated castor oil',
 'mentha arvensis leaf extract',
 'hydroxyacetophenone',
 'colloidal oatmeal',
 'myristic acid',
 'hydroxypropyl bislauramide mea',
 'isoceteth-20',
 'rosa damascena flower',
 'isopropyl palmitate',
 'menthyl lactate',
 'tetrasodium edta',
 'oxidized glutathione',
 'sodium sulfite',
 'nicotiana benthamiana hexapeptide-40 sh-oligopeptide-1',
 'iron oxides',
 'snail secretion filtrate',
 'polyglyceryl-3 methylglucose distearate',
 'propylene glycol',
 'rosa canina s

In [75]:
synonym_map = {

    # Water
    "aqua": "water",

    # Fragrance
    "parfum": "fragrance",
    "perfume": "fragrance",

    # Vitamin E
    "tocopheryl acetate": "vitamin e",

    # Vitamin B5
    "d-panthenol": "panthenol",

    # Hyaluronic Acid Family
    "hyaluronic acid": "sodium hyaluronate",

    # Vitamin C Family (keep separate if you want detailed analysis)
    # "ascorbic acid": "vitamin c",
    # "ethylated ascorbic acid": "vitamin c",
    # "ascorbyl glucoside": "vitamin c",

    # Niacinamide
    "vitamin b3": "niacinamide",

    # Zinc PCA
    "zinc pca ": "zinc pca",

    # Ceramides (OPTIONAL - keep separate initially)
    # "ceramide np": "ceramide",
    # "ceramide ap": "ceramide",
    # "ceramide eop": "ceramide",
}

#### Ingredient frequency analysis

In [114]:
from collections import Counter

In [77]:
ingredient_counter = Counter()
for ingredients in df["cleaned_ingredients"]:
    ingredient_counter.update(ingredients)

#### most common ingredients

In [78]:
def normalize_ingredient(ingredient):

    ingredient = ingredient.strip().lower()

    return synonym_map.get(ingredient, ingredient)

In [79]:
df["cleaned_ingredients"] = df["cleaned_ingredients"].apply(
    lambda ingredients: [
        normalize_ingredient(i)
        for i in ingredients
    ]
)

In [80]:
ingredient_counter.most_common(30)

[('xanthan gum', 74),
 ('glycerin', 71),
 ('sodium hyaluronate', 56),
 ('carbomer', 55),
 ('phenoxyethanol', 49),
 ('niacinamide', 45),
 ('water', 45),
 ('ethylhexylglycerin', 45),
 ('butylene glycol', 42),
 ('cholesterol', 41),
 ('caprylyl glycol', 41),
 ('phytosphingosine', 40),
 ('ceramide np', 39),
 ('1-2-hexanediol', 38),
 ('panthenol', 38),
 ('cetearyl alcohol', 38),
 ('sodium lauroyl lactylate', 38),
 ('propanediol', 38),
 ('ceramide ap', 35),
 ('ceramide eop', 34),
 ('sodium hydroxide', 33),
 ('trisodium ethylenediamine disuccinate', 33),
 ('aqua', 32),
 ('citric acid', 29),
 ('dimethicone', 29),
 ('disodium edta', 27),
 ('allantoin', 26),
 ('tocopherol', 25),
 ('glyceryl stearate', 24),
 ('chlorphenesin', 23)]

#### TF IDF 

In [81]:
df["ingredient_text"] = df["cleaned_ingredients"].apply(
    lambda x: " ".join(x)
)

In [82]:
df["ingredient_text"].head()

0    snail secretion filtrate butylene glycol niaci...
1    water cetearyl alcohol caprylic/​capric trigly...
2    water glycerin stearic acid myristic acid peg-...
3    glycolic acid water aloe barbadensis leaf wate...
4    water glycerin lactococcus ferment lysate copp...
Name: ingredient_text, dtype: str

#### BUILD TF IDF Matrix

In [83]:
import sys

!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [84]:

from sklearn.feature_extraction.text import TfidfVectorizer

In [85]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    df["ingredient_text"]
)

In [86]:
tfidf_matrix.shape

(130, 672)

#### Compute cosine similarity

In [87]:
from sklearn.metrics.pairwise import cosine_similarity

In [88]:
similarity_matrix = cosine_similarity(
    tfidf_matrix
)

In [89]:
similarity_matrix.shape

(130, 130)

#### Recommendation function for cosine similarity

In [90]:
def recommend(product_name, top_n=5):

    idx = df[
        df["product_name"] == product_name
    ].index[0]

    scores = list(
        enumerate(similarity_matrix[idx])
    )

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in scores[1:top_n+1]:

        recommendations.append(
            (
                df.iloc[i]["product_name"],
                round(score, 3)
            )
        )

    return recommendations

In [91]:
recommend(
    "The Ordinary Alpha Arbutin 2 % + Ha"
)

[('The Ordinary Alpha Arbutin 2% + Ha', np.float64(0.464)),
 ('The Ordinary Ascorbic Acid 8% + Alpha Arbutin 2%', np.float64(0.369)),
 ('COSRX 5 Pdrn B5 Vital Soothing Toner', np.float64(0.359)),
 ('COSRX 5 Pdrn Collagen Intense Vitalizing Serum', np.float64(0.34)),
 ('The Ordinary 2% Salicylic Acid Solution (2022)', np.float64(0.31))]

#### Jaccard similarity

In [92]:
def jaccard_similarity(list1, list2):
    
    set1 = set(list1)
    set2 = set(list2)
    
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    
    return intersection / union

In [93]:
jaccard_similarity(
    df["cleaned_ingredients"][0],
    df["cleaned_ingredients"][1]
)

0.1590909090909091

In [94]:
 df["product_name"].head(20)

0            COSRX Advanced Snail Radiance Dual Essence
1                       CeraVe Baby Moisturizing Lotion
2             COSRX Ac Collection Calming Foam Cleanser
3     The Ordinary Aha 30% + Bha 2% Peeling Solution...
4              The Ordinary Buffet + Copper Peptides 1%
5                  COSRX Balancium B5 D-Panthenol Cream
6           COSRX Advanced Snail 96 Mucin Power Essence
7                          The Ordinary GF 15% Solution
8              The Ordinary 100% L-Ascorbic Acid Powder
9     CeraVe Blemish Control Gel Moisturiser With 2%...
10          COSRX Balancium Comfort Ceramide Cream Mist
11                         COSRX B5 D-Panthenol Ampoule
12                                      CeraVe Cleanser
13              The Ordinary Glucoside Foaming Cleanser
14                       CeraVe Acne Foaming Cream Wash
15                   The Ordinary Alpha Arbutin 2% + Ha
16                 COSRX 5 Pdrn B5 Vital Soothing Toner
17           COSRX Aloe Soothing Sun Cream Spf50

#### First recommendation function

In [95]:
def recommend_similar_products(product_index, df):
    
    target_ingredients = df.iloc[product_index]["cleaned_ingredients"]
    
    similarities = []
    
    for i in range(len(df)):
        
        if i == product_index:
            continue
        
        score = jaccard_similarity(
            target_ingredients,
            df.iloc[i]["cleaned_ingredients"]
        )
        
        similarities.append((df.iloc[i]["product_name"], score))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:5]

In [96]:
recommend_similar_products(5, df)

[('COSRX B5 D-Panthenol Ampoule', 0.30303030303030304),
 ('COSRX Balancium Comfort Ceramide Cream (Discontinued)',
  0.28888888888888886),
 ('COSRX Balancium Comfort Ceramide Cream', 0.26666666666666666),
 ('COSRX Balancium Comfort Ceramide Hand Cream Intense', 0.2653061224489796),
 ('COSRX Advanced Snail Peptide Eye Cream', 0.25)]

In [97]:
import random
import pandas as pd

sample_products = random.sample(
    list(df["product_name"]),
    10
)

results = []

for product in sample_products:

    recommendations = recommend(product)

    for rec_name, score in recommendations:

        results.append({
            "query_product": product,
            "recommended_product": rec_name,
            "similarity_score": score
        })

evaluation_df = pd.DataFrame(results)

evaluation_df.head()

,query_product,recommended_product,similarity_score
0,The Ordinary Azelaic Acid Suspension 10%,The Ordinary Concealer (3.2N),0.452
1,The Ordinary Azelaic Acid Suspension 10%,The Ordinary Coverage Foundation,0.408
2,The Ordinary Azelaic Acid Suspension 10%,The Ordinary Ascorbyl Glucoside Solution 12%,0.331
3,The Ordinary Azelaic Acid Suspension 10%,The Ordinary Argireline Solution 10%,0.307
4,The Ordinary Azelaic Acid Suspension 10%,The Ordinary Alpha Arbutin 2 % + Ha,0.249


In [98]:
import random

sample_products = random.sample(
    list(df["product_name"]),
    10
)

for product in sample_products:

    print("=" * 80)
    print(f"QUERY PRODUCT: {product}")
    print("-" * 80)

    recommendations = recommend(product)

    for rec_name, score in recommendations:
        print(f"{rec_name} --> Similarity: {score}")

    print("\n")

QUERY PRODUCT: The Ordinary Glucoside Foaming Cleanser
--------------------------------------------------------------------------------
CeraVe Baby Wash & Shampoo [CAN] --> Similarity: 0.402
CeraVe Baby Wash & Shampoo --> Similarity: 0.38
The Ordinary Caffeine Solution 5% + Egcg --> Similarity: 0.318
The Ordinary Behentrimonium Chloride 2% Conditioner --> Similarity: 0.244
The Ordinary Caffeine Solution 5% + EGCG --> Similarity: 0.241


QUERY PRODUCT: CeraVe Acne Foaming Cream Wash
--------------------------------------------------------------------------------
CeraVe Acne Foaming Cleanser --> Similarity: 0.837
CeraVe Acne Foaming Skin Cleanser --> Similarity: 0.793
CeraVe Acne Foam Cream Cleanser --> Similarity: 0.693
CeraVe Anti-dandruff Hydrating Shampoo --> Similarity: 0.504
CeraVe Acne Control Gel --> Similarity: 0.497


QUERY PRODUCT: COSRX Aloe Soothing Sun Cream Spf50+
--------------------------------------------------------------------------------
COSRX Aloe Soothing Sun Cream

#### categorising products

In [99]:
def categorize_product(product_name):

    name = product_name.lower()

    # Most specific first

    if "eye patch" in name:
        return "eye patch"

    elif "acne patch" in name or "barrier patches" in name:
        return "patch"

    elif "sun stick" in name:
        return "sun stick"

    elif "sunscreen" in name or "spf" in name:
        return "sunscreen"

    elif "shampoo" in name:
        return "shampoo"

    elif "conditioner" in name:
        return "conditioner"

    elif "cleansing balm" in name:
        return "cleanser"

    elif "cleanser" in name:
        return "cleanser"

    elif "limpiador" in name:
        return "cleanser"

    elif "remover" in name:
        return "makeup remover"

    elif "foundation" in name or "concealer" in name:
        return "makeup"

    elif "ointment" in name:
        return "ointment"

    elif "moisturizer" in name:
        return "moisturizer"

    elif "moisturising lotion" in name:
        return "lotion"

    elif "lotion" in name:
        return "lotion"

    elif "cream" in name:
        return "cream"

    elif "mask" in name:
        return "mask"

    elif "toning solution" in name:
        return "toner"

    elif "toner" in name:
        return "toner"

    elif "ampoule" in name:
        return "ampoule"

    elif "essence" in name:
        return "essence"

    elif "gel" in name:
        return "gel"

    elif "peeling solution" in name:
        return "exfoliant"

    elif "glycolic acid" in name:
        return "exfoliant"

    elif "bha" in name:
        return "exfoliant"

    elif "aha" in name:
        return "exfoliant"

    elif "solution" in name:
        return "serum"

    elif "retinoid" in name:
        return "serum"

    elif "retinol" in name:
        return "serum"

    elif "niacinamide" in name:
        return "serum"

    elif "alpha arbutin" in name:
        return "serum"

    elif "buffet" in name:
        return "serum"

    elif "argireline" in name:
        return "serum"

    elif "peptide" in name:
        return "serum"

    elif "amino acids" in name:
        return "serum"

    elif "azelaic acid" in name:
        return "serum"

    elif "oil" in name or "squalane" in name:
        return "oil"

    elif "powder" in name:
        return "powder"

    elif "serum" in name:
        return "serum"
    
    elif "liquid" in name:
        return "toner"
    
    elif "vitamin c" in name:
        return "serum"
    
    elif "alpha lipoic acid" in name:
        return "serum"
    
    elif "euk" in name:
        return "serum"
    
    elif "crema" in name:
        return "cream"

    else:
        return "other"

In [100]:
df["category"] = df["product_name"].apply(categorize_product)

In [101]:
df["category"].value_counts()

category
serum             30
cleanser          16
cream             15
sunscreen         14
oil               10
exfoliant          8
toner              5
lotion             4
ointment           4
essence            3
patch              3
shampoo            3
gel                2
mask               2
eye patch          2
makeup             2
conditioner        2
powder             1
ampoule            1
sun stick          1
makeup remover     1
moisturizer        1
Name: count, dtype: int64

In [102]:
df[df["category"] == "other"]["product_name"].tolist()

[]

#### modufied recommendation function

In [103]:
def recommend_by_category(product_name, top_n=5):

    query_idx = df[df["product_name"] == product_name].index[0]

    query_category = df.loc[query_idx, "category"]

    category_df = df[df["category"] == query_category]

    recommendations = []

    for idx in category_df.index:

        if idx == query_idx:
            continue

        score = similarity_matrix[query_idx][idx]

        recommendations.append({
            "recommended_product":
                df.loc[idx, "product_name"],

            "category":
                df.loc[idx, "category"],

            "similarity":
                round(float(score), 3)
        })

    recommendations = sorted(
        recommendations,
        key=lambda x: x["similarity"],
        reverse=True
    )

    return pd.DataFrame(
        recommendations[:top_n]
    )

In [104]:
recommend_by_category(
    "COSRX Advanced Snail Radiance Dual Essence"
)

,recommended_product,category,similarity
0,COSRX Advanced Snail 96 Mucin Power Essence,essence,0.411
1,COSRX Advanced Snail 96 Mucin Power Essence (E...,essence,0.403


In [105]:
recommend_by_category(

    "The Ordinary Alpha Arbutin 2% + Ha"

)

,recommended_product,category,similarity
0,The Ordinary Alpha Arbutin 2 % + Ha,serum,0.464
1,The Ordinary Buffet,serum,0.383
2,The Ordinary Ascorbic Acid 8% + Alpha Arbutin 2%,serum,0.320
3,COSRX 5 Pdrn Collagen Intense Vitalizing Serum,serum,0.236
4,The Ordinary Azelaic Acid Suspension 10%,serum,0.189


#### Add frequency based ingredient weightage

#### calculate ingredient document frequency

In [106]:
from collections import defaultdict

ingredient_df = defaultdict(int)

for ingredients in df["cleaned_ingredients"]:
    
    unique_ingredients = set(ingredients)
    
    for ingredient in unique_ingredients:
        ingredient_df[ingredient] += 1

#### calculate weightage

In [107]:
import math

N = len(df)

ingredient_weights = {}

for ingredient, freq in ingredient_df.items():
    
    ingredient_weights[ingredient] = math.log(N / freq)

#### check top weighted ingredients

In [108]:
sorted(

    ingredient_weights.items(),

    key=lambda x: x[1],

    reverse=True

)[:20]

[('limnanthes alba seed oil', 4.867534450455582),
 ('sodium surfactin', 4.867534450455582),
 ('arginine pca', 4.867534450455582),
 ('oleic acid', 4.867534450455582),
 ('arachidic acid', 4.867534450455582),
 ('raspberry ketone', 4.867534450455582),
 ('benzyl glycol', 4.867534450455582),
 ('hydrolyzed glycosaminoglycans', 4.867534450455582),
 ('sorbitan sesquiisostearate', 4.867534450455582),
 ('sorbeth-30 tetraisostearate', 4.867534450455582),
 ('acrylates/\u200bbeheneth-25 methacrylate copolymer', 4.867534450455582),
 ('ppg-8-ceteth-20', 4.867534450455582),
 ('methyl gluceth-10', 4.867534450455582),
 ('diphenyl dimethicone', 4.867534450455582),
 ('panthenol 30.0%', 4.867534450455582),
 ('edetate disodium', 4.867534450455582),
 ('ammonium acryloyldimethyltaurate/\u200bvp copolymer', 4.867534450455582),
 ('t-butyl alcohol', 4.867534450455582),
 ('citronellol', 4.867534450455582),
 ('geraniol', 4.867534450455582)]

#### check lowest weighted ingredients

In [109]:
sorted(

    ingredient_weights.items(),

    key=lambda x: x[1]

)[:20]

[('water', 0.5368011101692514),
 ('xanthan gum', 0.5634693572514127),
 ('glycerin', 0.604854573414267),
 ('sodium hyaluronate', 0.7086513670959106),
 ('carbomer', 0.8602012652231116),
 ('phenoxyethanol', 0.9757141523449557),
 ('niacinamide', 1.0608719606852626),
 ('ethylhexylglycerin', 1.0608719606852626),
 ('butylene glycol', 1.1298648321722142),
 ('cholesterol', 1.1539623837512747),
 ('caprylyl glycol', 1.1539623837512747),
 ('phytosphingosine', 1.1786549963416462),
 ('ceramide np', 1.2039728043259361),
 ('panthenol', 1.2299482907291965),
 ('1-2-hexanediol', 1.2299482907291965),
 ('sodium lauroyl lactylate', 1.2299482907291965),
 ('cetearyl alcohol', 1.2299482907291965),
 ('propanediol', 1.2299482907291965),
 ('ceramide ap', 1.3121863889661687),
 ('ceramide eop', 1.341173925839421)]

In [110]:
for ingredient in sorted(vocab):

    if (
        "read" in ingredient
        or "detail" in ingredient
        or ">>" in ingredient
    ):
        print(ingredient)

#### Longest ingredient names

In [112]:
sorted(vocab, key=len, reverse=True)[:30]

['bis-behenyl/\u200bisostearyl/\u200bphytosteryl dimer dilinoleyl dimer dilinoleate',
 'hydroxyethyl acrylate/\u200bsodium acryloyldimethyl taurate copolymer',
 'polyglyceryl-4 diisostearate/\u200bpolyhydroxystearate/\u200bsebacate',
 'hydrogenated dimer dilinoleyl/\u200bdimethylcarbonate copolymer',
 'vinyl dimethicone/\u200bmethicone silsesquioxane crosspolymer',
 'nicotiana benthamiana hexapeptide-40 sh-oligopeptide-1',
 'nicotiana benthamiana hexapeptide-40 sh-polypeptide-76',
 'pentaerythrityl tetra-di-t-butyl hydroxyhydrocinnamate',
 'nicotiana benthamiana octapeptide-30 sh-oligopeptide-2',
 'paraffinum liquidum/\u200bmineral oil/\u200bhuile minerale',
 'dipeptide diaminobutyroyl benzylamide diacetate',
 'simmondsia chinensis seed oil/\u200bjojoba seed oil',
 'ethylbisiminomethylguaiacol manganese chloride',
 'ammonium acryloyldimethyltaurate/\u200bvp copolymer',
 'bis-ethylhexyloxyphenol methoxyphenyl triazine',
 'acrylates/\u200bc10-30 alkyl acrylate crosspolymer',
 'prunus amy

#### Check Single-Occurrence Ingredients

In [117]:
from collections import Counter

counter = Counter()

for ingredients in df["cleaned_ingredients"]:
    counter.update(ingredients)

In [118]:
rare_ingredients = [
    ingredient
    for ingredient, count in counter.items()
    if count == 1
]

print(f"Number of ingredients appearing once: {len(rare_ingredients)}")

rare_ingredients[:50]

Number of ingredients appearing once: 206


['limnanthes alba seed oil',
 'sodium surfactin',
 'arginine pca',
 'arachidic acid',
 'oleic acid',
 'hydrolyzed glycosaminoglycans',
 'benzyl glycol',
 'raspberry ketone',
 'methyl gluceth-10',
 'sorbeth-30 tetraisostearate',
 'sorbitan sesquiisostearate',
 'acrylates/\u200bbeheneth-25 methacrylate copolymer',
 'ppg-8-ceteth-20',
 'panthenol 30.0%',
 'diphenyl dimethicone',
 'edetate disodium',
 'ammonium acryloyldimethyltaurate/\u200bvp copolymer',
 't-butyl alcohol',
 'hexyl cinnamal',
 'benzyl salicylate',
 'geraniol',
 'citronellol',
 'benzyl benzoate',
 'rosa damascena flower water',
 'rosa damascena flower',
 'hexylene glycol',
 'tamarindus indica seed gum',
 'illicium verum fruit extract',
 'scutellaria baicalensis root extract',
 'hydrogenated poly',
 'hydrogenated styrene/\u200bbutadiene copolymer',
 'paraffinum liquidum/\u200bmineral oil/\u200bhuile minerale',
 'meradimate',
 'alumina',
 'aluminum stearate',
 'caprylic/\u200bcapric triglycerides',
 'polypropylsilsesquioxane

#### ingredients containing numbers

In [119]:
for ingredient in sorted(vocab):

    if any(char.isdigit() for char in ingredient):

        print(ingredient)

1-2-hexanediol
2- hexanediol
3-butanediol
3-o-ethyl ascorbic acid
acetyl hexapeptide-8
acrylates/​beheneth-25 methacrylate copolymer
acrylates/​c10-30 alkyl acrylate crosspolymer
alpha-arbutin 2.0%
ascorbic acid 8.0%
azelaic acid 10.0%
biosaccharide gum-2
c12-14 pareth-12
c12-15 alkyl benzoate
c12-20 alkyl glucoside
c13-16 isoparaffin
c14-22 alcohols
ceramide 1
ceramide 3
ceramide 6 ii
ceramide 6-ii
ceteareth-12
ceteareth-20
ceteth-20
ceteth-25
ci 77491
ci 77492
copper tripeptide-1
dimethicone/​bis-isobutyl ppg-20 crosspolymer
dimethicone/​peg-10/​15 crosspolymer
ethyl ascorbic acid 15.0%
isoceteth-20
laureth-4
methyl gluceth-10
nicotiana benthamiana hexapeptide-40 sh-oligopeptide-1
nicotiana benthamiana hexapeptide-40 sh-polypeptide-76
nicotiana benthamiana octapeptide-30 sh-oligopeptide-2
oleth-25
palmitoyl pentapeptide-4
palmitoyl tetrapeptide-7
palmitoyl tripeptide-1
palmitoyl tripeptide-38
panthenol 30.0%
peg-100
peg-100 stearate
peg-120 methyl glucose dioleate
peg-150 pentaerythr

#### Ingredients ending with punctuation

In [120]:
for ingredient in sorted(vocab):

    if ingredient.endswith("."):

        print(ingredient)